In [ ]:
import torch
import pickle
import os

from kgate.knowledgegraph import KnowledgeGraph as KGATEKnowledgeGraph
from my_knowledge_graph import KnowledgeGraph as TKGEKnowledgeGraph


def torchkge_to_kgate(tkge_kg):
    df = tkge_kg.get_df().rename(columns={'from': 'head', 'to': 'tail', 'rel': 'edge'})
    kg = KGATEKnowledgeGraph(
        dataframe=df,
        node_to_index=tkge_kg.ent2ix,
        edge_to_index=tkge_kg.rel2ix
    )
    kg.removed_triplets = torch.zeros((4, 0), dtype=torch.long)
    return kg


def build_removed_triplets_from_dicts(tkge_kg):
    heads, tails, edges = [], [], []
    for (h, r), true_tails in tkge_kg.dict_of_tails.items():
        for t in true_tails:
            heads.append(h)
            tails.append(t)
            edges.append(r)
    n = len(heads)
    return torch.stack([
        torch.tensor(heads, dtype=torch.long),
        torch.tensor(tails, dtype=torch.long),
        torch.tensor(edges, dtype=torch.long),
        torch.zeros(n, dtype=torch.long)
    ], dim=0)


def load_tkge_splits(input_path):
    """Load TorchKGE splits from .pt or .pkl"""
    if os.path.isdir(input_path):
        kg_train = torch.load(os.path.join(input_path, 'kg_train.pt'), weights_only=False)
        kg_val   = torch.load(os.path.join(input_path, 'kg_val.pt'),   weights_only=False)
        kg_test  = torch.load(os.path.join(input_path, 'kg_test.pt'),  weights_only=False)
    elif input_path.endswith('.pkl'):
        with open(input_path, 'rb') as f:
            kg_train = pickle.load(f)
            kg_val   = pickle.load(f)
            kg_test  = pickle.load(f)
    else:
        raise ValueError(f"Format not recognized: {input_path}")
    return kg_train, kg_val, kg_test


def convert_and_save(input_path, output_dir, verify=True):
    """
    Convert TorchKGE splits to KGATE and save.
    
    input_path : dir containing kg_train.pt/kg_val.pt/kg_test.pt, or a single .pkl file
    output_dir : output dir
    verify     : verbose
    """
    print(f"Loading file: {input_path}")
    kg_train_tkge, kg_val_tkge, kg_test_tkge = load_tkge_splits(input_path)
    assert kg_train_tkge.ent2ix == kg_val_tkge.ent2ix == kg_test_tkge.ent2ix
    assert kg_train_tkge.rel2ix == kg_val_tkge.rel2ix == kg_test_tkge.rel2ix
    print(f"train: {kg_train_tkge.n_facts} triples")
    print(f"val:   {kg_val_tkge.n_facts} triples")
    print(f"test:  {kg_test_tkge.n_facts} triples")

    print("Converting to KGATE...")
    kg_train = torchkge_to_kgate(kg_train_tkge)
    kg_val   = torchkge_to_kgate(kg_val_tkge)
    kg_test  = torchkge_to_kgate(kg_test_tkge)

    kg_train.removed_triplets = build_removed_triplets_from_dicts(kg_train_tkge)

    if verify:
        full_graphindices = torch.cat([
            kg_train.graphindices, kg_train.removed_triplets,
            kg_val.graphindices,   kg_val.removed_triplets,
            kg_test.graphindices,  kg_test.removed_triplets
        ], dim=1)
        n_dict   = sum(len(v) for v in kg_train_tkge.dict_of_tails.values())
        n_unique = torch.unique(full_graphindices[:3], dim=1).shape[1]
        print(f"True positives in dict_of_tails : {n_dict}")
        print(f"Unique triples in full_graphindices : {n_unique}")
        print(f"Match : {n_dict == n_unique}")

    os.makedirs(output_dir, exist_ok=True)
    torch.save(kg_train, os.path.join(output_dir, 'kg_train.pt'))
    torch.save(kg_val,   os.path.join(output_dir, 'kg_val.pt'))
    torch.save(kg_test,  os.path.join(output_dir, 'kg_test.pt'))
    print(f"Saved in {output_dir}")

    return kg_train, kg_val, kg_test

In [ ]:
kg_train, kg_val, kg_test = convert_and_save(
    input_path='/home/galadriel/dr_benchmark/DL1experiment/withDL1/kg_processing/',
    output_dir='/home/galadriel/dr_benchmark/DL1experiment/withDL1/kg_processing/kgate/'
)
